# PSSN Week 2 — Build the semantic-state dataset

Run after notebooks 07 and 08. This joins the original baseline traces, whole validated state sequences, and W2-005 vector index. It stores state text and vector links in JSONL; vectors remain in their separate NumPy file. Every run writes to a new timestamped folder and leaves raw artifacts unchanged.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
from google.colab import drive
drive.mount("/content/drive")

try:
    from src.data.semantic_state_dataset import build_semantic_state_dataset, read_jsonl, save_semantic_state_dataset
    print("W2-006 dataset builder is available.")
except ImportError:
    from google.colab import files
    uploaded = files.upload()
    name = "semantic_state_dataset.py"
    matches = [content for uploaded_name, content in uploaded.items() if Path(uploaded_name).name == name]
    if len(matches) != 1:
        raise ValueError("Upload exactly src/data/semantic_state_dataset.py from the repository.")
    destination = Path("src/data/semantic_state_dataset.py")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(matches[0])
    from src.data.semantic_state_dataset import build_semantic_state_dataset, read_jsonl, save_semantic_state_dataset
    print("Uploaded the W2-006 dataset builder.")
print("Repository:", Path.cwd())

In [ ]:
from datetime import datetime, timezone
import hashlib, json

ARTIFACT_ROOT = Path("/content/drive/MyDrive/PSSN2/artifacts/expanded_baseline")
candidate_configs = sorted(ARTIFACT_ROOT.glob("*/semantic_state_segmentation/*/quality_validation/*/representations/*/representation_config.json"))
if not candidate_configs:
    raise FileNotFoundError("No W2-005 representation_config.json found. Run notebook 08 first.")
representation_config_path = candidate_configs[-1]
representation_dir = representation_config_path.parent
validation_dir = representation_dir.parent.parent
run_dir = validation_dir.parents[3]
week3_path = validation_dir / "week3_ready_states.jsonl"
reports_path = validation_dir / "trace_validation_reports.jsonl"
validation_summary_path = validation_dir / "validation_summary.json"
baseline_path = run_dir / "segmentation_ready_traces.jsonl"
representation_index_path = representation_dir / "representation_index.jsonl"
vector_path = representation_dir / "state_vectors.npy"
for required_path in (week3_path, reports_path, validation_summary_path, baseline_path, representation_index_path, vector_path):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required W2 input is missing: {required_path}")
representation_config = json.loads(representation_config_path.read_text(encoding="utf-8"))
baseline_records = read_jsonl(baseline_path)
week3_states = read_jsonl(week3_path)
validation_reports = read_jsonl(reports_path)
representation_index = read_jsonl(representation_index_path)
validation_summary = json.loads(validation_summary_path.read_text(encoding="utf-8"))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

seed_values = {record.get("run_metadata", {}).get("seed") for record in baseline_records if record.get("run_metadata", {}).get("seed") is not None}
if len(seed_values) > 1:
    raise ValueError(f"Baseline records contain inconsistent random seeds: {sorted(seed_values)}")
random_seed = next(iter(seed_values), 42)
source_paths = {
    "baseline_traces": baseline_path,
    "week3_ready_states": week3_path,
    "state_validation_reports": reports_path,
    "state_validation_summary": validation_summary_path,
    "representation_index": representation_index_path,
    "representation_config": representation_config_path,
    "state_vectors": vector_path,
}
source_files = {key: {"path": str(path), "sha256": sha256_file(path)} for key, path in source_paths.items()}
dataset_records, dataset_metadata, dataset_summary = build_semantic_state_dataset(
    baseline_records, week3_states, validation_reports, representation_index,
    representation_config=representation_config,
    vector_file=vector_path,
    random_seed=random_seed,
    validation_thresholds=validation_summary.get("thresholds", {}),
    source_files=source_files,
)
dataset_metadata["source_config_path"] = str(representation_config_path)
dataset_metadata["repository_commit"] = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or None
dataset_metadata["dataset_builder_sha256"] = sha256_file(Path("src/data/semantic_state_dataset.py"))
dataset_metadata["created_at_utc"] = datetime.now(timezone.utc).isoformat()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = run_dir / "semantic_state_dataset" / stamp
output_paths = save_semantic_state_dataset(output_dir, dataset_records, dataset_metadata, dataset_summary)
print(json.dumps(dataset_summary, indent=2))
print("Saved dataset files:")
for name, path in output_paths.items():
    print(f"{name}: {path}")

In [ ]:
saved_records = read_jsonl(output_paths["dataset"])
assert len(saved_records) == dataset_summary["included_trace_count"]
assert sum(record["state_count"] for record in saved_records) == dataset_summary["included_state_count"]
assert all(state["representation"].get("vector_file") for record in saved_records for state in record["states"])
assert all(not any(key.lower().startswith(("black_hole", "low_value", "high_value")) for key in record) for record in saved_records)
print(f"Verified {len(saved_records)} trace records and {dataset_summary['included_state_count']} vector-linked states.")
print("First record fields:", sorted(saved_records[0]))

## Optional: publish a private copy to Hugging Face
Run this cell only when ready to upload. It creates or reuses ramitha2002/pssn-sementic-state-math-problems as a private dataset repo and prompts for a write token using hidden input (not saved in the notebook). Uploads are blocked if the repo is public. The upload includes this dataset card at the repo root and the final JSONL, metadata, summary, and vector matrix under a timestamped runs/ folder.

In [ ]:
from getpass import getpass
from tempfile import TemporaryDirectory
import shutil
from huggingface_hub import HfApi

HF_DATASET_REPO_ID = "ramitha2002/pssn-sementic-state-math-problems"
HF_TOKEN = getpass("Hugging Face write token (input hidden): ").strip()
if not HF_TOKEN:
    raise ValueError("A Hugging Face write token is required.")

hf_api = HfApi(token=HF_TOKEN)
hf_api.create_repo(repo_id=HF_DATASET_REPO_ID, repo_type="dataset", private=True, exist_ok=True)
repo_info = hf_api.repo_info(repo_id=HF_DATASET_REPO_ID, repo_type="dataset")
if not repo_info.private:
    raise PermissionError("This Hugging Face dataset repository is public. Make it private before uploading research data.")

with TemporaryDirectory(prefix="pssn_hf_dataset_") as temp_dir:
    publish_dir = Path(temp_dir)
    run_publish_dir = publish_dir / "runs" / output_dir.name
    run_publish_dir.mkdir(parents=True)
    publish_records_path = run_publish_dir / "semantic_states.jsonl"
    with publish_records_path.open("w", encoding="utf-8", newline="\n") as stream:
        for record in saved_records:
            for state in record["states"]:
                state["representation"]["vector_file"] = "state_vectors.npy"
            stream.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n")
    publish_metadata = json.loads(Path(output_paths["metadata"]).read_text(encoding="utf-8"))
    publish_metadata["dataset_name"] = "PSSN-sementic-state-math-problems"
    publish_metadata["source_vector_file"] = publish_metadata.get("vector_file")
    publish_metadata["vector_file"] = "state_vectors.npy"
    (run_publish_dir / "semantic_state_metadata.json").write_text(json.dumps(publish_metadata, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    shutil.copy2(output_paths["summary"], run_publish_dir / "semantic_state_summary.json")
    shutil.copy2(vector_path, run_publish_dir / "state_vectors.npy")
    shutil.copy2(Path("huggingface_dataset/README.md"), publish_dir / "README.md")
    commit = hf_api.upload_folder(
        folder_path=str(publish_dir),
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        commit_message="Upload PSSN semantic-state dataset " + output_dir.name,
    )

del HF_TOKEN
print("Uploaded private dataset:", "https://huggingface.co/datasets/" + HF_DATASET_REPO_ID + "/tree/main")
print("Uploaded run:", "runs/" + output_dir.name)
print("Commit:", commit.oid)